In [1]:
import sys, os
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import f1_score

sys.path.insert(0, '../src')
from dataset import EmotionDataset, load_split, EMOTION_COLS
from model import MultiLabelEmotionClassifier

CKPT_PATH  = '../models/BETO_best.pt'
HF_NAME    = 'dccuchile/bert-base-spanish-wwm-cased'
DEV_PATH   = '../dev/dev.csv'
TEST_PATH  = '../test/test.csv'

if torch.backends.mps.is_available(): DEVICE = torch.device('mps')
elif torch.cuda.is_available():        DEVICE = torch.device('cuda')
else:                                  DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
thresholds = ckpt.get('thresholds', np.full(len(EMOTION_COLS), 0.5))
print(f'Checkpoint cargado. Thresholds: {dict(zip(EMOTION_COLS, thresholds.round(2)))}')

tokenizer = AutoTokenizer.from_pretrained(HF_NAME)
model = MultiLabelEmotionClassifier(HF_NAME, dropout=0.3).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()

def predict(df):
    loader = DataLoader(EmotionDataset(df, tokenizer, 256),
                        batch_size=16, shuffle=False)
    logits_list = []
    labels_list = []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            tt   = batch.get('token_type_ids')
            if tt is not None: tt = tt.to(DEVICE)
            logits_list.append(model(ids, mask, tt).cpu().numpy())
            if 'labels' in batch:
                labels_list.append(batch['labels'].numpy())
    logits = np.vstack(logits_list)
    labels = np.vstack(labels_list) if labels_list else None
    probs  = 1 / (1 + np.exp(-logits))
    preds  = (probs >= thresholds).astype(int)
    return preds, labels

dev_df  = load_split(DEV_PATH)
test_df = load_split(TEST_PATH)

dev_preds,  dev_labels  = predict(dev_df)
test_preds, test_labels = predict(test_df)

dev_f1  = f1_score(dev_labels,  dev_preds,  average='micro', zero_division=0)
test_f1 = f1_score(test_labels, test_preds, average='micro', zero_division=0)

print(f'\nDev  Micro F1 : {dev_f1:.4f}')
print(f'Test Micro F1 : {test_f1:.4f}')
print()
print('Esperado → Dev≈0.4566 | Test≈0.4851')
if abs(test_f1 - 0.4851) < 0.005:
    print('✓ Checkpoint correcto')
else:
    print('✗ Checkpoint distinto al esperado')


Device: mps
Checkpoint cargado. Thresholds: {'anger': np.float64(0.35), 'fear': np.float64(0.8), 'joy': np.float64(0.5), 'sadness': np.float64(0.85), 'surprise': np.float64(0.4), 'hope': np.float64(0.15)}


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Dev  Micro F1 : 0.4567
Test Micro F1 : 0.4058

Esperado → Dev≈0.4566 | Test≈0.4851
✗ Checkpoint distinto al esperado
